# ORFMOS

**Permanent Colab Front Door**

This notebook is intentionally thin. It resolves the mutable ORFMOS entry selector, verifies the immutable host-entry chain, runs host authorization, and then transfers execution to the permanent GitHub durable bootstrap.

Human entry is stable; machine authority remains SHA256.


In [ ]:
import hashlib
import json
import urllib.parse
import urllib.request
from datetime import datetime, timezone

ORFMOS_FRONT_DOOR_SCHEMA = "ORFMOS_COLAB_FRONT_DOOR_V0_1"
ORFMOS_FRONT_DOOR_VERSION = "0.1"
ORFMOS_FRONT_DOOR_REVISION = "V0_1_STABLE_COLAB_URL_SELECTOR_ENTRY_MANIFEST"

ORFMOS_ENTRY_SELECTOR_URL = "https://raw.githubusercontent.com/hadrex83/ORFMOS/main/boot/ORFMOS_ENTRY.json"
ORFMOS_ALLOWED_REPOSITORY = "hadrex83/ORFMOS"
ORFMOS_ALLOWED_HOST = "raw.githubusercontent.com"
ORFMOS_FRONT_DOOR_TIMEOUT_SECONDS = 30
ORFMOS_FRONT_DOOR_MAX_SELECTOR_BYTES = 65536
ORFMOS_FRONT_DOOR_MAX_MANIFEST_BYTES = 1048576
ORFMOS_FRONT_DOOR_MAX_COMPONENT_BYTES = 33554432

def _orfmos_front_utc():
    return datetime.now(timezone.utc).isoformat()

def _orfmos_front_sha(raw):
    return hashlib.sha256(raw).hexdigest()

def _orfmos_front_validate_url(url):
    parsed = urllib.parse.urlsplit(str(url or "").strip())
    if parsed.scheme != "https":
        raise RuntimeError("FRONT_DOOR_URL_HTTPS_REQUIRED")
    if parsed.hostname != ORFMOS_ALLOWED_HOST:
        raise RuntimeError("FRONT_DOOR_URL_HOST_NOT_ALLOWED")
    prefix = "/" + ORFMOS_ALLOWED_REPOSITORY + "/"
    if not parsed.path.startswith(prefix):
        raise RuntimeError("FRONT_DOOR_URL_REPOSITORY_NOT_ALLOWED")
    if parsed.username or parsed.password or parsed.port:
        raise RuntimeError("FRONT_DOOR_URL_AUTHORITY_NOT_ALLOWED")
    return parsed

def _orfmos_front_fetch(url, max_bytes):
    _orfmos_front_validate_url(url)
    req = urllib.request.Request(
        url,
        headers={
            "Accept": "application/octet-stream",
            "User-Agent": "ORFMOS-Colab-Front-Door/0.1",
            "Cache-Control": "no-cache",
        },
        method="GET",
    )
    with urllib.request.urlopen(req, timeout=ORFMOS_FRONT_DOOR_TIMEOUT_SECONDS) as response:
        status = int(getattr(response, "status", 200) or 200)
        if status != 200:
            raise RuntimeError("FRONT_DOOR_HTTP_STATUS_" + str(status))
        final_url = str(getattr(response, "geturl", lambda: url)() or url)
        _orfmos_front_validate_url(final_url)
        raw = response.read(max_bytes + 1)
    if len(raw) > max_bytes:
        raise RuntimeError("FRONT_DOOR_OBJECT_TOO_LARGE")
    return raw

def _orfmos_front_json(raw, schema):
    try:
        obj = json.loads(raw.decode("utf-8"))
    except Exception as exc:
        raise RuntimeError("FRONT_DOOR_JSON_INVALID:" + type(exc).__name__) from exc
    if not isinstance(obj, dict):
        raise RuntimeError("FRONT_DOOR_JSON_OBJECT_REQUIRED")
    if obj.get("schema") != schema:
        raise RuntimeError("FRONT_DOOR_SCHEMA_MISMATCH")
    return obj

def _orfmos_front_validate_sha(value, label):
    value = str(value or "").lower()
    if len(value) != 64 or any(ch not in "0123456789abcdef" for ch in value):
        raise RuntimeError(label + "_SHA_INVALID")
    return value

def orfmos_front_door():
    print("=" * 112)
    print("ORFMOS FRONT DOOR v0.1 — COLAB HOST ENTRY / VERIFIED GITHUB RESOLVER")
    print("=" * 112)
    print("Entry selector :", ORFMOS_ENTRY_SELECTOR_URL)
    print("Authority      : SELECTOR -> IMMUTABLE ENTRY MANIFEST -> VERIFIED COMPONENTS")
    print("Write authority: NONE")
    print("-" * 112)

    selector_raw = _orfmos_front_fetch(
        ORFMOS_ENTRY_SELECTOR_URL,
        ORFMOS_FRONT_DOOR_MAX_SELECTOR_BYTES,
    )
    selector_sha = _orfmos_front_sha(selector_raw)
    selector = _orfmos_front_json(selector_raw, "ORFMOS_ENTRY_SELECTOR_V0_1")

    if selector.get("state") != "ACTIVE":
        raise RuntimeError("FRONT_DOOR_SELECTOR_NOT_ACTIVE")
    if selector.get("provider") != "GITHUB_RAW":
        raise RuntimeError("FRONT_DOOR_PROVIDER_UNSUPPORTED")
    if selector.get("repository") != ORFMOS_ALLOWED_REPOSITORY:
        raise RuntimeError("FRONT_DOOR_REPOSITORY_MISMATCH")

    manifest_record = selector.get("manifest")
    if not isinstance(manifest_record, dict):
        raise RuntimeError("FRONT_DOOR_MANIFEST_RECORD_REQUIRED")
    manifest_url = str(manifest_record.get("url") or "")
    _orfmos_front_validate_url(manifest_url)
    manifest_expected_sha = _orfmos_front_validate_sha(
        manifest_record.get("sha256"), "FRONT_DOOR_MANIFEST"
    )
    manifest_expected_bytes = manifest_record.get("bytes")
    if not isinstance(manifest_expected_bytes, int) or manifest_expected_bytes <= 0:
        raise RuntimeError("FRONT_DOOR_MANIFEST_BYTES_INVALID")
    if manifest_record.get("publication_state") != "IMMUTABLE":
        raise RuntimeError("FRONT_DOOR_MANIFEST_NOT_IMMUTABLE")

    manifest_raw = _orfmos_front_fetch(
        manifest_url,
        ORFMOS_FRONT_DOOR_MAX_MANIFEST_BYTES,
    )
    manifest_sha = _orfmos_front_sha(manifest_raw)
    if len(manifest_raw) != manifest_expected_bytes:
        raise RuntimeError("FRONT_DOOR_MANIFEST_BYTES_MISMATCH")
    if manifest_sha != manifest_expected_sha:
        raise RuntimeError("FRONT_DOOR_MANIFEST_SHA_MISMATCH")

    manifest = _orfmos_front_json(
        manifest_raw, "ORFMOS_COLAB_ENTRY_MANIFEST_V0_1"
    )
    if manifest.get("publication_state") != "IMMUTABLE":
        raise RuntimeError("FRONT_DOOR_ENTRY_MANIFEST_NOT_IMMUTABLE")
    if manifest.get("host_provider") != "COLAB":
        raise RuntimeError("FRONT_DOOR_HOST_PROVIDER_MISMATCH")
    if manifest.get("loader_semantics") != "exec(compile(...), globals(), globals())":
        raise RuntimeError("FRONT_DOOR_LOADER_SEMANTICS_MISMATCH")

    components = manifest.get("components")
    if not isinstance(components, list) or len(components) != 2:
        raise RuntimeError("FRONT_DOOR_COMPONENTS_INVALID")

    staged = []
    seen_orders = set()
    for row in sorted(components, key=lambda x: int(x.get("order", -1))):
        if not isinstance(row, dict):
            raise RuntimeError("FRONT_DOOR_COMPONENT_RECORD_INVALID")
        order = row.get("order")
        role = str(row.get("role") or "")
        url = str(row.get("url") or "")
        expected_bytes = row.get("bytes")
        expected_sha = _orfmos_front_validate_sha(
            row.get("sha256"), "FRONT_DOOR_COMPONENT"
        )
        if not isinstance(order, int) or order in seen_orders:
            raise RuntimeError("FRONT_DOOR_COMPONENT_ORDER_INVALID")
        if role not in ("HOST_AUTHORIZATION_GATE", "DURABLE_BOOTSTRAP"):
            raise RuntimeError("FRONT_DOOR_COMPONENT_ROLE_INVALID")
        if not isinstance(expected_bytes, int) or expected_bytes <= 0:
            raise RuntimeError("FRONT_DOOR_COMPONENT_BYTES_INVALID")
        _orfmos_front_validate_url(url)
        seen_orders.add(order)

        raw = _orfmos_front_fetch(url, ORFMOS_FRONT_DOOR_MAX_COMPONENT_BYTES)
        observed_sha = _orfmos_front_sha(raw)
        if len(raw) != expected_bytes:
            raise RuntimeError("FRONT_DOOR_COMPONENT_BYTES_MISMATCH:" + role)
        if observed_sha != expected_sha:
            raise RuntimeError("FRONT_DOOR_COMPONENT_SHA_MISMATCH:" + role)
        compile(raw, url, "exec")  # syntax gate before any component execution
        staged.append((row, raw))
        print(
            f"VALIDATED {order:03d} {role} · "
            f"{len(raw)} bytes · {observed_sha[:16]}…"
        )

    roles = [row["role"] for row, _ in staged]
    if roles != ["HOST_AUTHORIZATION_GATE", "DURABLE_BOOTSTRAP"]:
        raise RuntimeError("FRONT_DOOR_COMPONENT_SEQUENCE_INVALID")

    globals()["__ORFMOS_FRONT_DOOR__"] = {
        "schema": ORFMOS_FRONT_DOOR_SCHEMA,
        "state": "VALIDATED",
        "front_door_version": ORFMOS_FRONT_DOOR_VERSION,
        "front_door_revision": ORFMOS_FRONT_DOOR_REVISION,
        "entry_selector_url": ORFMOS_ENTRY_SELECTOR_URL,
        "entry_selector_sha256": selector_sha,
        "entry_selector_generation": selector.get("selector_generation"),
        "entry_manifest_url": manifest_url,
        "entry_manifest_sha256": manifest_sha,
        "component_count": len(staged),
        "host_provider": "COLAB",
        "loader_semantics": "exec(compile(...), globals(), globals())",
        "write_authority": "NONE",
        "validated_utc": _orfmos_front_utc(),
    }

    # Execute the certified host authorization gate first.
    gate_row, gate_raw = staged[0]
    print(f"LOAD      {gate_row['order']:03d} {gate_row['role']}")
    exec(compile(gate_raw, gate_row["url"], "exec"), globals(), globals())

    # Only after the host gate returns do we execute the durable bootstrap.
    boot_row, boot_raw = staged[1]
    print(f"LOAD      {boot_row['order']:03d} {boot_row['role']}")
    exec(compile(boot_raw, boot_row["url"], "exec"), globals(), globals())

    boot = globals().get("__ORFMOS_DURABLE_BOOT_ANCHOR__")
    if not isinstance(boot, dict) or boot.get("state") != "COMPLETE":
        raise RuntimeError("FRONT_DOOR_DURABLE_BOOT_NOT_COMPLETE")

    record = globals().get("__ORFMOS_FRONT_DOOR__")
    record["state"] = "COMPLETE"
    record["bios_version"] = boot.get("bios_version")
    record["bios_revision"] = boot.get("bios_revision")
    record["boot_selector_generation"] = boot.get("selector_generation")
    record["completed_utc"] = _orfmos_front_utc()

    print("-" * 112)
    print("FRONT DOOR STATE: COMPLETE")
    print("BIOS            :", boot.get("bios_version"), "/", boot.get("bios_revision"))
    print("=" * 112)
    return record

orfmos_front_door()
